# Vergleichende Analyse des Token-Längen-Experiments (Metrik: 256 vs. 512 vs. 1024 Tokens)

Dieses Notebook analysiert und visualisiert den Einfluss unterschiedlicher maximaler Token-Sequenzlängen (**256**, **512**, **1024**) auf die **Simplicity-Metrik-Modelle (BiLSTM MixUp Regressoren)**:
1. **Statistische Evaluation auf dem Lebenshilfe-Datensatz:** Vorhersagewerte auf Leichter Sprache (LS) vs. Alltagssprache (AS), Trennschärfe (Separation Margin) und Klassifikationsgenauigkeit.
2. **Längen-Bias & Längenstabilität:** Korrelation zwischen Textlänge und Vorhersagewert sowie stratifizierte Analyse nach Textlängen-Klassen.
3. **KDE-Dichteverteilungs-Plots:** Darstellung der Verteilungen (Grün für LS, Blau für AS) im Format der Referenzabbildung.

---

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

def find_repo_root():
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, "data")) and os.path.exists(os.path.join(p, "results")):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.path.expanduser("~/Documents/Master Thesis"))

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("Arbeitsverzeichnis:", os.getcwd())

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "figure.dpi": 150
})

METRIC_SUMMARY_CSV = os.path.join(REPO_ROOT, "results/evaluation/token_length_metric_comparison.csv")
METRIC_DETAILS_CSV = os.path.join(REPO_ROOT, "results/evaluation/token_length_metric_details.csv")


## 1. Statistische Auswertung auf dem Lebenshilfe-Datensatz

In [ ]:
df_details = pd.read_csv(METRIC_DETAILS_CSV)
df_summary = pd.read_csv(METRIC_SUMMARY_CSV)

records = []
for model_name, label in [
    ("metric_mixup_256", "BiLSTM MixUp (256 Tokens)"),
    ("metric_mixup_512", "BiLSTM MixUp (512 Tokens)"),
    ("metric_mixup_1024", "BiLSTM MixUp (1024 Tokens)")
]:
    sub = df_details[df_details["metric_model"] == model_name]
    if len(sub) == 0:
        continue
    records.append({
        "Modell / Kontextlänge": label,
        "Mean Score (LS)": f"{sub['score_ls'].mean():.3f} ± {sub['score_ls'].std():.3f}",
        "Mean Score (AS)": f"{sub['score_as'].mean():.3f} ± {sub['score_as'].std():.3f}",
        "Separation Margin (LS - AS)": f"{sub['margin'].mean():+.3f}",
        "Genauigkeit (LS > AS)": f"{sub['correct_order'].mean() * 100:.1f} %",
        "Val Loss (MSE)": f"{df_summary.loc[df_summary['metric_model'] == model_name, 'val_mse'].values[0]:.4f}" if "val_mse" in df_summary.columns else "-"
    })

df_comp = pd.DataFrame(records)
print(f"Geladene Stichproben: {len(df_details)} Detailsätze aus {METRIC_DETAILS_CSV}")
display(df_comp)


## 2. Kernmetriken im Vergleich (LS- vs. AS-Scores und Separation Margin)

In [ ]:
df_m = pd.read_csv(METRIC_SUMMARY_CSV)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scores auf LS vs AS
df_melt = pd.melt(df_m, id_vars=["metric_model", "max_seq_len"], value_vars=["mean_score_ls", "mean_score_as"],
                  var_name="Textkorpus", value_name="Mittlerer Score")
df_melt["Textkorpus"] = df_melt["Textkorpus"].map({"mean_score_ls": "Leichte Sprache (LS)", "mean_score_as": "Alltagssprache (AS)"})
df_melt["Token-Länge"] = df_melt["max_seq_len"].astype(str) + " Tokens"

sns.barplot(data=df_melt, x="Token-Länge", y="Mittlerer Score", hue="Textkorpus", ax=axes[0], palette=["#2ecc71", "#3498db"])
axes[0].set_title("Mittlere Modellvorhersage auf LS vs. AS")
axes[0].set_ylim(0, 1.05)

# Separation Margin
sns.barplot(data=df_m, x="max_seq_len", y="separation_margin", ax=axes[1], palette="viridis", hue="max_seq_len", legend=False)
axes[1].set_title("Separation Margin (Score_LS - Score_AS)")
axes[1].set_xlabel("Maximale Token-Länge")
axes[1].set_ylabel("Mittlere Differenz")

plt.tight_layout()
plot_out1 = os.path.join(REPO_ROOT, "results/plots/experiments/token_length/token_length_metric_scores_margin.png")
os.makedirs(os.path.dirname(plot_out1), exist_ok=True)
plt.savefig(plot_out1, dpi=300, bbox_inches="tight")
print(f"Plot gespeichert unter: {plot_out1}")
plt.show()


## 3. Stratifizierte Trennschärfe nach Textlänge der Artikel
Untersuchung, wie gut die Modelle auf kurzen (< 200 Tokens), mittleren (200-450 Tokens) und langen (> 450 Tokens) Artikeln trennen.

In [ ]:
df_m = pd.read_csv(METRIC_SUMMARY_CSV)
df_strat = pd.melt(df_m, id_vars=["max_seq_len"], value_vars=["margin_short", "margin_med", "margin_long"],
                   var_name="Längenklasse", value_name="Separation Margin")
df_strat["Längenklasse"] = df_strat["Längenklasse"].map({
    "margin_short": "Kurz (<200 Tokens)",
    "margin_med": "Mittel (200-450 Tokens)",
    "margin_long": "Lang (>450 Tokens)"
})
df_strat["Token-Länge"] = df_strat["max_seq_len"].astype(str) + " Tokens"

plt.figure(figsize=(10, 5))
sns.barplot(data=df_strat, x="Längenklasse", y="Separation Margin", hue="Token-Länge", palette="magma")
plt.title("Separation Margin nach Textlänge des Ausgangstextes (AS)")
plt.ylabel("Mittlere Differenz (Score_LS - Score_AS)")
plt.xlabel("Länge des AS-Artikels")
plt.legend(title="Modell-Kontextlänge")
plt.tight_layout()
plot_out2 = os.path.join(REPO_ROOT, "results/plots/experiments/token_length/token_length_metric_stratified.png")
plt.savefig(plot_out2, dpi=300, bbox_inches="tight")
print(f"Plot gespeichert unter: {plot_out2}")
plt.show()


## 4. KDE-Plots der Dichteverteilungen (256 vs. 512 vs. 1024 Tokens)
Visuelle Gegenüberstellung der drei Token-Längen:
- **Grüne Fläche:** `LS (Leichte Sprache)`
- **Blaue Fläche:** `AS (Alltagssprache)`
- **X-Achse:** `Lambda (Anteil LS)`
- **Y-Achse:** `Dichte`

In [ ]:
df_details = pd.read_csv(METRIC_DETAILS_CSV)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

model_configs = [
    ("metric_mixup_256", "BiLSTM MixUp (256 Tokens)"),
    ("metric_mixup_512", "BiLSTM MixUp (512 Tokens)"),
    ("metric_mixup_1024", "BiLSTM MixUp (1024 Tokens)")
]

for idx, (m_name, title) in enumerate(model_configs):
    sub = df_details[df_details["metric_model"] == m_name]
    ax = axes[idx]
    sns.kdeplot(sub["score_ls"], fill=True, color="#2ecc71", label="Leichte Sprache (LS)", ax=ax, alpha=0.4)
    sns.kdeplot(sub["score_as"], fill=True, color="#3498db", label="Alltagssprache (AS)", ax=ax, alpha=0.4)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Modell-Vorhersage (Simplicity Score)")
    ax.set_ylabel("Dichte")
    ax.set_xlim(-0.05, 1.05)
    ax.legend(loc="upper center")

plt.suptitle("Dichteverteilungen der Metrik-Modelle auf dem Lebenshilfe Benchmark", fontsize=14, fontweight="bold")
plt.tight_layout()
plot_out3 = os.path.join(REPO_ROOT, "results/plots/experiments/token_length/token_length_metric_kde.png")
plt.savefig(plot_out3, dpi=300, bbox_inches="tight")
print(f"Plot gespeichert unter: {plot_out3}")
plt.show()
